# JARVIS BRAIN v1 — Qwen3-4B Baseline + SFT Experiment (Kaggle GPU)
#
**Architecture**
#
```
JARVIS PC            Kaggle GPU          Hugging Face
   |                   |                    |
Model Gateway   -> JARVIS Brain      Durable Artifacts
   |            -> Qwen3-4B           (model / adapter /
   |                   |                 config / dataset / logs)
```
#
**Rules**
* Do NOT full-pretrain Qwen3-4B.
* Do NOT store weights in the JARVIS source repo.
* Do NOT call Qwen "Silverwing" — Silverwing is the project's own model family.
* Publish to immutable revisions/tags. Keep checkpoints/experiments in a separate staging namespace.
* Do not promote the model until the evaluation gate passes.
#
**Job**
Env audit → Auth → Download → Validate → **Baseline benchmark** → Dataset prep → Small SFT (LoRA) → Eval → Publish adapter/artifacts → Register in JARVIS Model Registry.

## 1. Environment audit — hardware reality
Do NOT assume T4/P100. Print the ACTUAL accelerator.

In [ ]:
import platform, psutil, shutil, os, json, time, sys, traceback
import torch
import transformers

print('hostname     :', platform.node())
print('python       :', platform.python_version())
print('torch        :', torch.__version__)
print('transformers :', transformers.__version__)
print('cuda built   :', torch.version.cuda)
print('cuda avail   :', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  gpu[{i}]: {p.name} | {p.total_memory/1e9:.1f} GB VRAM | cuda {torch.version.cuda}')
else:
    print('  no CUDA GPU detected (CPU-only run)')
print('cpu cores    :', psutil.cpu_count(logical=False))
print('system RAM   : {:.1f} GB'.format(psutil.virtual_memory().total/1e9))
print('disk free    : {:.1f} GB'.format(shutil.disk_usage('/kaggle').free/1e9))
print('bf16 support :', bool(torch.cuda.is_bf16_supported()) if torch.cuda.is_available() else 'n/a (cpu)')

ACCEL = {
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'gpu_count': torch.cuda.device_count() if torch.cuda.is_available() else 0,
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'vram_gb': round(torch.cuda.get_device_properties(0).total_memory/1e9, 1) if torch.cuda.is_available() else None,
    'ram_gb': round(psutil.virtual_memory().total/1e9, 1),
    'bf16': bool(torch.cuda.is_bf16_supported()) if torch.cuda.is_available() else False,
}
print('\nACCEL summary:', json.dumps(ACCEL, indent=2))

## 2. Authentication
Set a Kaggle Secret named `JARVIS_HF_TOKEN` (or `HF_TOKEN`).

In [ ]:
HF_TOKEN = os.environ.get('JARVIS_HF_TOKEN') or os.environ.get('HF_TOKEN') or ''
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN: configured ({})'.format(len(HF_TOKEN)))
else:
    print('WARN: no HF token set. Public downloads will still work.')

from huggingface_hub import HfApi, model_info, upload_folder
API = HfApi(token=HF_TOKEN or None)
print('HF user:', (API.whoami().get('name') if HF_TOKEN else 'anonymous'))

## 3. Verify model access + exact revision (before any download)

In [ ]:
MODEL_ID = 'Qwen/Qwen3-4B'
REVISION = '1cfa9a7208912126459214e8b04321603b3df60c'  # pinned immutable revision
INFO = model_info(MODEL_ID, revision=REVISION)
print('repo       :', INFO.id)
print('revision   :', INFO.sha)
print('license    :', INFO.license or '(unknown)')
print('tags       :', [t for t in INFO.tags if t.startswith(('arxiv', 'license', 'base_'))])
print('files      :', len(INFO.siblings))
assert INFO.sha.startswith(REVISION[:8]), 'revision mismatch'
cfg_path = API.hf_hub_download(MODEL_ID, 'config.json', revision=REVISION)
config = json.load(open(cfg_path, encoding='utf-8'))
print('arch       :', config['architectures'])
print('model_type :', config['model_type'])
print('ctx        :', config['max_position_embeddings'])
print('layers     :', config['num_hidden_layers'])

## 4. Inference feasibility — memory benchmark
4B bf16 ≈ 8 GB weights. On a 16 GB T4 that fits with FP16 weights; on 8 GB it does not → use 4-bit (QLoRA-grade) load. Decide ADAPTER_MODE accordingly.

In [ ]:
ADAPTER_MODE = 'lora'
if torch.cuda.is_available():
    vram = ACCEL['vram_gb']
    if vram is not None and vram < 12:
        print(f'VRAM {vram} GB < 12 GB -> use 4-bit base + LoRA (QLoRA-style load).')
        LOAD_BITS, ADAPTER_MODE = 4, 'qlo'
    else:
        print(f'VRAM {vram} GB >= 12 GB -> bf16 base + LoRA feasible.')
        LOAD_BITS, ADAPTER_MODE = 16, 'lora'
else:
    print('No GPU -> inference/SFT not feasible here; set SMALL_MODE to run a sanity check on CPU only.')
    LOAD_BITS = 32
print('LOAD_BITS =', LOAD_BITS, '| ADAPTER_MODE =', ADAPTER_MODE)

## 5. Slow mode (optional) — small sanity run on CPU

In [ ]:
SMALL_MODE = os.environ.get('SMALL_MODE', '0') == '1'
MAX_SEQ = int(os.environ.get('MAX_SEQ', '2048'))
MAX_BATCH = int(os.environ.get('MAX_BATCH', '4'))
EPOCHS = float(os.environ.get('EPOCHS', '1.0'))
MAX_STEPS = int(os.environ.get('MAX_STEPS', '0'))  # 0 => full epochs
RANK = int(os.environ.get('LORA_RANK', '16'))
print(f'SMALL_MODE={SMALL_MODE} MAX_SEQ={MAX_SEQ} MAX_BATCH={MAX_BATCH} EPOCHS={EPOCHS} MAX_STEPS={MAX_STEPS} RANK={RANK}')

## 6. Download model (weights NOT in JARVIS repo — cached in Kaggle working dir)

In [ ]:
from transformers import AutoConfig, AutoTokenizer
from pathlib import Path
WORK = Path('/kaggle/working')
MODEL_CACHE = WORK / 'model_cache'
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(WORK / 'hf_home'))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION, trust_remote_code=False, use_fast=True)
print('tokenizer vocab:', len(tokenizer))
tok_cache = MODEL_CACHE / 'tokenizer'
tokenizer.save_pretrained(tok_cache)
print('tokenizer cached ->', tok_cache)

## 7. Baseline inference benchmark — Qwen3-4B UNTUNED (before ANY JARVIS customization)Runs the pinned revision on the exact same fixed prompt set + deterministic scorer used for the Silverwing-120M comparison runner (rain_benchmarks.py), so the PC-side jarvis_brain_comparison.json is a fair A/B.Saves results to /kaggle/working/qwen3_4b_baseline.json as the reference point.

In [ ]:
import torch as _t
_KERNEL_ROOT = Path('/kaggle/working/..') if Path('/kaggle/working/..').exists() else None
try:
    _KERNEL_DIR = Path(__file__).resolve().parent
except NameError:
    _KERNEL_DIR = _KERNEL_ROOT
if _KERNEL_DIR is not None:
    sys.path.insert(0, str(_KERNEL_DIR))
try:
    from brain_benchmarks import run_qwen3_4b
    print('shared benchmark harness: ok')
except Exception as _exc:
    print('shared harness unavailable (%s) — falling back to inline probes.' % _exc)
    run_qwen3_4b = None

BASELINE_PROBES = {
    'general-conversation': "Introduce yourself briefly and say what you can help with.",
    'math': "Solve for x: 2x + 6 = 14. Show your working.",
    'logic': "If all fruits are foods and an apple is a fruit, is an apple a food? Answer yes or no and explain in one sentence.",
    'coding': "Write a Python function is_valid_ipv4(s) that returns True only for valid dotted-quad IPv4 addresses.",
    'knowledge': "What is 15% of 200?",
    'instruction-following': "Reply with exactly one word: the capital of France.",
}

def _baseline_generate(gen_prompt: str, max_new=128) -> str:
    msgs = [{"role": "user", "content": gen_prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt").input_ids.to(base.device)
    with _t.inference_mode():
        out_ids = base.generate(
            ids, max_new_tokens=max_new, do_sample=True,
            temperature=0.7, top_p=0.9, top_k=20, pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out_ids[0][ids.shape[1]:], skip_special_tokens=True).strip()

print(f'Baseline benchmark on {MODEL_ID}@{REVISION[:8]} ...')
if run_qwen3_4b is not None:
    BASELINE_AGG = run_qwen3_4b(_baseline_generate, tokenizer=tokenizer)
    BASELINE_RESULTS = {r['category']: r for r in BASELINE_AGG['results']}
    for cat, agg in sorted(BASELINE_AGG['categories'].items()):
        print(f'  [{cat}] score={agg} tok/s={BASELINE_AGG["tokens_per_sec_avg"]} -> {BASELINE_RESULTS[cat]["answer"][:70]!r}')
else:
    BASELINE_RESULTS = {}
    for cat, prompt in BASELINE_PROBES.items():
        try:
            t0 = time.time()
            answer = _baseline_generate(prompt)
            dt = time.time() - t0
            BASELINE_RESULTS[cat] = {
                'prompt': prompt, 'answer': answer,
                'latency_s': round(dt, 2),
                'new_tokens': None, 'tokens_per_sec': None,
            }
            print(f'  [{cat}] -> {answer[:80]!r}')
        except Exception as exc:
            BASELINE_RESULTS[cat] = {'error': str(exc)}
            print(f'  [{cat}] FAILED: {exc}')
            if ACCEL['device'] == 'cpu' and ACCEL['ram_gb'] < 10:
                print('   -> cpu/ram constraint confirmed; baseline limited.')

BASELINE_FILE = WORK / 'qwen3_4b_baseline.json'
BASELINE_FILE.write_text(json.dumps({
    'model': MODEL_ID, 'revision': REVISION, 'license': INFO.license,
    'accelerator': ACCEL,
    'harness': 'brain_benchmarks.py (fixed set + deterministic scorer)',
    'aggregate': BASELINE_AGG if run_qwen3_4b is not None else {c: r for c, r in BASELINE_RESULTS.items()},
}, indent=2), encoding='utf-8')
print('saved baseline ->', BASELINE_FILE)



## 8. Dataset: JARVIS Qwen3 SFT (from repo, no vendor lock-in)
Load `jarvis-brain-qwen3-4b-sft-v1` dataset (normalized `messages` format). Fall back to HF dataset repo if it exists.

In [ ]:
DATASET_REPO = 'videlisndichi/jarvis-brain-qwen3-4b-sft-v1'   # <- publish target
DATASET_SEED_REPO = 'videlisndichi/jarvis-skills-sft'          # 403 existing examples
from datasets import load_dataset, Dataset, concatenate_datasets

def _load_messages(repo, split='train', revision='main'):
    try:
        ds = load_dataset(repo, split=split, revision=revision)
        if 'messages' not in ds.column_names and 'text' in ds.column_names:
            # convert text into messages when possible
            def to_mes(r):
                s = r.get('text', '').strip()
                if not s:
                    return None
                return {'messages': [
                    {'role': 'system', 'content': 'You are JARVIS, an intelligent assistant.'},
                    {'role': 'user', 'content': s.split('[SEP]')[0] if '[SEP]' in s else s},
                    {'role': 'assistant', 'content': '[SEP]'.join(s.split('[SEP]')[1:]) if '[SEP]' in s else ''},
                ]}
            ds = ds.map(to_mes, remove_columns=ds.column_names)
            ds = ds.filter(lambda r: r.get('messages') is not None and r['messages'][-1]['content'])
        return ds
    except Exception as e:
        print('  fallback: could not load', repo, '->', e)
        return None

samples = []

# --- primary expanded dataset
ds = _load_messages(DATASET_REPO)
if ds is not None:
    print('loaded expanded:', DATASET_REPO, len(ds))
    samples.append(ds)
else:
    # --- seed dataset
    ds = _load_messages(DATASET_SEED_REPO)
    if ds is not None:
        print('loaded seed:', DATASET_SEED_REPO, len(ds))
        samples.append(ds)

if not samples:
    raise RuntimeError('No dataset found; publish expanded dataset first (pc dataset builder).')
DATA = concatenate_datasets(samples)
print('combined samples:', len(DATA))
print('sample keys:', DATA.column_names)
print('example:', json.dumps(DATA[0], ensure_ascii=False)[:300])

## 9. Train/dev split (structured, no leakage by source)

In [ ]:
from datasets import DatasetDict

def split_by_mix(ds, val_frac=0.1, seed=7):
    all_ds = ds.shuffle(seed=seed)
    n = len(all_ds)
    val = int(n * val_frac)
    return DatasetDict({'train': all_ds.select(range(val, n)), 'dev': all_ds.select(range(val))})

SPLIT = split_by_mix(DATA)
print('train:', len(SPLIT['train']), 'dev:', len(SPLIT['dev']))
DEV = SPLIT['dev']
TRAIN = SPLIT['train']

## 10. Formatting for Qwen3 chat (thinking + tool templates optional)

In [ ]:
def fmt_messages(messages):
    # Qwen3 chat template with <|im_start|>/<|im_end|>
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)

def preprocess(example):
    text = fmt_messages(example['messages'])
    enc = tokenizer(text, truncation=True, max_length=MAX_SEQ, padding=False)
    enc['labels'] = enc['input_ids'].copy()
    return enc

TRAIN_F = TRAIN.map(preprocess, remove_columns=TRAIN.column_names, batched=False, desc='fmt train')
DEV_F = DEV.map(preprocess, remove_columns=DEV.column_names, batched=False, desc='fmt dev')
print('train tokenized:', len(TRAIN_F), '| max len used:', max(len(x['input_ids']) for x in TRAIN_F))

## 11. Load base model — frozen, LoRA on top (NEVER overwrite base)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_kw = {}
if LOAD_BITS == 4:
    bnb_kw['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    bnb_kw['device_map'] = 'auto'

print(f'Loading {MODEL_ID} (bits={LOAD_BITS}) — this downloads ~8 GB once into Kaggle disk cache ...')
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION,
    torch_dtype=torch.bfloat16 if LOAD_BITS == 16 else torch.float32,
    trust_remote_code=False, use_cache=False,
    **bnb_kw,
)
print(f'base loaded in {time.time()-t0:.1f}s | dtype={base.dtype} | params={base.num_parameters()/1e9:.2f}B')
for p in base.parameters():
    p.requires_grad = False
base_arch = base.config.architectures[0]

## 12. PEFT / LoRA adapter — frozen base, train only LoRA params

In [ ]:
try:
    import peft
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
    PEFT_OK = True
    print('peft', peft.__version__)
except Exception as e:
    print('no peft ->', e); PEFT_OK = False

if PEFT_OK:
    if LOAD_BITS == 4:
        base = prepare_model_for_kbit_training(base)
    lora_cfg = LoraConfig(
        r=RANK, lora_alpha=2 * RANK, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    )
    model = get_peft_model(base, lora_cfg)
    trainable, allp = model.get_nb_trainable_parameters()
    print(f'LoRA trainable: {trainable/1e6:.1f}M / {allp/1e9:.2f}B ({100*trainable/allp:.2f}%)')
else:
    model = base
    print('peft unavailable -> training skipped')

## 13. Training — small SFT experiment (frozen base + LoRA adapter)

In [ ]:
import numpy as np
from transformers import (DataCollatorForSeq2Seq, Trainer, TrainingArguments,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding='longest', pad_to_multiple_of=8)
if os.environ.get('SMALL_MODE') == '1' and not PEFT_OK:
    print('SMALL_MODE CPU sanity: shaking the trainer object only; NO actual training on CPU.')
    TRAIN_ACTIVE = False
else:
    TRAIN_ACTIVE = PEFT_OK and torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=str(WORK / 'lora_out'),
    per_device_train_batch_size=MAX_BATCH,
    per_device_eval_batch_size=MAX_BATCH,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS else None,
    bf16=LOAD_BITS == 16,
    fp16=LOAD_BITS == 4,
    fp16_full_eval=False,
    logging_steps=10,
    eval_strategy='steps' if TRAIN_ACTIVE else 'no',
    eval_steps=100,
    save_strategy='steps', save_steps=100, save_total_limit=2,
    report_to=[],  # no wandb/neptune
    gradient_checkpointing=LOAD_BITS == 4,
    optim='paged_adamw_8bit' if LOAD_BITS == 4 else 'adamw_torch',
)

if TRAIN_ACTIVE:
    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=TRAIN_F,
        eval_dataset=DEV_F,
        data_collator=collator,
    )
    print('training start ...')
    t0 = time.time()
    trainer.train()
    print(f'trained in {time.time()-t0:.1f}s')
    metrics = trainer.evaluate()
    print('eval:', json.dumps(metrics, indent=2))
else:
    print('Skipping actual training (no GPU or peft missing). Baseline-only run.')

## 14. Save adapter + baseline/eval artifacts (staging dir)

In [ ]:
STAGE = WORK / 'jarvis-brain-qwen3-4b-sft-v1'
STAGE.mkdir(parents=True, exist_ok=True)
if PEFT_OK and TRAIN_ACTIVE:
    model.save_pretrained(str(STAGE))
    tokenizer.save_pretrained(str(STAGE))
    print('adapter staged ->', STAGE)
else:
    # baseline run artifacts
    (STAGE / 'README.md').write_text(
        '# jarvis-brain-qwen3-4b-sft-v1\n\nBaseline-only run '
        '(no GPU / peft unavailable). No adapter produced.\n', encoding='utf-8')
    print('no adapter produced (baseline-only). staged README.')

RUN_JSON = STAGE / 'run_manifest.json'
manifest = {
    'experiment': 'jarvis-brain-qwen3-4b-sft-v1',
    'base_model': MODEL_ID, 'base_revision': REVISION, 'base_license': INFO.license,
    'adapter_mode': ADAPTER_MODE, 'load_bits': LOAD_BITS,
    'accelerator': ACCEL, 'trained': TRAIN_ACTIVE,
    'dataset_repo': DATASET_REPO, 'num_train': len(TRAIN),
    'max_seq': MAX_SEQ, 'epochs': EPOCHS, 'steps': MAX_STEPS,
    'lora_rank': RANK, 'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
RUN_JSON.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('run_manifest.json:\n', json.dumps(manifest, indent=2))

## 15. Publish to Hugging Face — durable artifacts only (never base weights)

In [ ]:
if HF_TOKEN and PEFT_OK and TRAIN_ACTIVE:
    print('Publishing adapter/artifacts to', f'{DATASET_REPO}@<tag> ...')
    tag = time.strftime('sft-%Y%m%d-%H%M', time.gmtime())
    API.create_repo(repo_id=DATASET_REPO, exist_ok=True, private=False)
    upload_folder(folder_path=str(STAGE), repo_id=DATASET_REPO, token=HF_TOKEN,
                  commit_message=f'JARVIS Brain SFT experiment {tag}')
    # immutable tag on the released repo
    try:
        API.create_tag(repo_id=DATASET_REPO, tag=tag, ref='main')
        print('released tag ->', tag)
    except Exception as e:
        print('tag warn:', e)
else:
    print('Publication skipped (need HF token + trained adapter).')

## 16. Registry entry for JARVIS Model Registry (copy into catalog)

In [ ]:
registry_entry = {
    'model_id': 'jarvis-brain-qwen3-4b-v1' if not (PEFT_OK and TRAIN_ACTIVE) else 'jarvis-brain-qwen3-4b-sft-v1',
    'status': 'evaluated' if (PEFT_OK and TRAIN_ACTIVE) else 'experimental',
    'base_model': MODEL_ID, 'base_revision': REVISION, 'license': INFO.license,
    'parameter_count': int(base.num_parameters()),
    'context_length': config['max_position_embeddings'],
    'vocab_size': config['vocab_size'],
    'capabilities': ['text-generation', 'chat', 'reasoning', 'coding', 'math', 'tool-use', 'long-context'],
    'serving': {'huggingface': DATASET_REPO},
    'deployment': 'kaggle-gpu',
    'evaluation': metrics if (PEFT_OK and TRAIN_ACTIVE) else 'pending-baseline',
}
print('REGISTRY_ENTRY:\n', json.dumps(registry_entry, indent=2))
REGISTRY_JSON = WORK / 'registry_entry.json'
REGISTRY_JSON.write_text(json.dumps(registry_entry, indent=2), encoding='utf-8')
print('written ->', REGISTRY_JSON)

## 17. Remote inference smoke test (OpenAI-compatible) — for the JARVIS Model Gateway
The final piece: the Kaggle host serves an OpenAI-compatible endpoint that JARVIS calls via `JARVIS_BRAIN_ENDPOINT`. This notebook stops at artifact publication; the gateway wiring is documented in `docs/MODEL_GATEWAY_REMOTE.md`.

In [ ]:
print('''DONE.

Remote hub (JARVIS PC) after this run:
1. Copy the REGISTRY_ENTRY JSON into silverwing-backend/model_catalog.py as
   "jarvis-jarvis-brain-qwen3-4b-v1" or "...-sft-v1".
2. Set JARVIS_BRAIN_ENDPOINT=<OpenAI-compatible URL served by the Kaggle GPU host>.
3. Gateway policy: general -> JARVIS Brain; offline -> silverwing-120m-base-v1; specialized see routing policy.
''')
assert not (HF_TOKEN and INFO.license != 'apache-2.0'), 'license check failed'
print('license OK:', INFO.license)